In [64]:
import sys
import subprocess

print("Installing required Python libraries...")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-U",
    "gspread",
    "google-auth",
    "openai",
    "python-dotenv"
])

print("Required Python libraries installed successfully.")

Installing required Python libraries...
Required Python libraries installed successfully.


In [65]:
import os
import json
import gspread
import google.auth

from decimal import Decimal, ROUND_HALF_UP
from openai import OpenAI
from dotenv import load_dotenv

print("Required libraries imported successfully.")

Required libraries imported successfully.


In [66]:
import os
import importlib
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    try:
        userdata = importlib.import_module("google.colab.userdata")
        OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    except:
        OPENROUTER_API_KEY = None

if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY was not found. "
        "Add it to the .env file in VS Code "
        "or to Colab Secrets."
    )

print("OpenRouter API key loaded successfully.")

OpenRouter API key loaded successfully.


In [4]:
import os
import glob
import gspread
import importlib

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

try:
    colab_auth = importlib.import_module("google.colab.auth")
    google_auth = importlib.import_module("google.auth")

    colab_auth.authenticate_user()

    creds, _ = google_auth.default(
        scopes=SCOPES
    )

    print("Google Colab authentication completed successfully.")

except ModuleNotFoundError:
    from google_auth_oauthlib.flow import InstalledAppFlow

    credentials_files = glob.glob(
        os.path.join("credentials", "*.json")
    )

    credentials_files = [
        file for file in credentials_files
        if not os.path.basename(file).startswith("token")
    ]

    if not credentials_files:
        raise FileNotFoundError(
            "No Google OAuth JSON file found in the credentials folder."
        )

    if len(credentials_files) > 1:
        raise ValueError(
            "Multiple Google OAuth JSON files found. "
            "Keep only one credentials JSON file in the credentials folder."
        )

    credentials_file = credentials_files[0]

    flow = InstalledAppFlow.from_client_secrets_file(
        credentials_file,
        SCOPES
    )

    creds = flow.run_local_server(port=0)

    print("Google Sheets authentication completed successfully.")

sheets_client = gspread.authorize(creds)

print("Google Sheets connection established successfully.")

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=699505262007-ohqtv468uuhbj013hlir07jmcgrfulft.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A62253%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fspreadsheets+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&state=9wmxEp7AyW8SdkL10tkt1BFD6VYdin&code_challenge=B8gjp0ExxiaNaMwtaxVcrjPDHq3JmSJ24L8DR2ne6tY&code_challenge_method=S256&access_type=offline
Google Sheets authentication completed successfully.
Google Sheets connection established successfully.


In [5]:
GOOGLE_SHEET_ID = "1QTHrfg9dMz2KeG29jYTwD0Vo0iJrr13f25CK8jSZ4J8"

if GOOGLE_SHEET_ID == "YOUR_GOOGLE_SHEET_ID":
    raise ValueError(
        "Please enter your Google Sheet ID in GOOGLE_SHEET_ID."
    )

spreadsheet = sheets_client.open_by_key(
    GOOGLE_SHEET_ID
)

payroll_sheet = spreadsheet.worksheet("Payroll")
attendance_sheet = spreadsheet.worksheet("Attendance")
sales_sheet = spreadsheet.worksheet("Sales")
attendance_policy_sheet = spreadsheet.worksheet("Attendance_Policy")
commission_policy_sheet = spreadsheet.worksheet("Commission_Policy")

print("Google Sheets workbook connected successfully.")
print("Required worksheets connected successfully.")

Google Sheets workbook connected successfully.
Required worksheets connected successfully.


In [57]:
payroll_data = payroll_sheet.get_all_records()
attendance_data = attendance_sheet.get_all_records()
sales_data = sales_sheet.get_all_records()
attendance_policy_data = attendance_policy_sheet.get_all_records()
commission_policy_data = commission_policy_sheet.get_all_records()

print("Payroll data loaded successfully.")
print("Attendance data loaded successfully.")
print("Sales data loaded successfully.")
print("Attendance policy data loaded successfully.")
print("Commission policy data loaded successfully.")

Payroll data loaded successfully.
Attendance data loaded successfully.
Sales data loaded successfully.
Attendance policy data loaded successfully.
Commission policy data loaded successfully.


In [59]:
from datetime import datetime, timedelta
import math

def parse_date(value):
    if value is None or str(value).strip() == "":
        return None

    if isinstance(value, datetime):
        return value.date()

    value = str(value).strip()

    for fmt in ("%Y-%m-%d", "%d-%b-%Y", "%d/%m/%Y"):
        try:
            return datetime.strptime(value, fmt).date()
        except:
            pass

    try:
        serial = float(value)
        return (datetime(1899, 12, 30) + timedelta(days=serial)).date()
    except:
        return None


def get_active_policy(policies, effective_date, policy_type=None, day_type=None):
    matches = []

    for policy in policies:
        if str(policy.get("Active", "")).strip().lower() != "yes":
            continue

        policy_date = parse_date(policy.get("Effective From"))

        if not policy_date or policy_date > effective_date:
            continue

        if policy_type and str(policy.get("Policy Type", "")).strip() != policy_type:
            continue

        if day_type and str(policy.get("Day Type", "")).strip() != day_type:
            continue

        matches.append((policy_date, policy))

    if not matches:
        return None

    matches.sort(key=lambda x: x[0], reverse=True)
    return matches[0][1]


def parse_percent(value):
    return float(str(value).replace("%", "").strip())


def calculate_commission(employee_id, start_date, end_date):
    total_sales = 0

    for sale in sales_data:
        if str(sale.get("Employee ID", "")).strip() != employee_id:
            continue

        sale_date = parse_date(sale.get("Sales Date"))

        if sale_date and start_date <= sale_date <= end_date:
            total_sales += float(sale.get("Sales Amount", 0) or 0)

    applicable_policy = None

    for policy in commission_policy_data:
        if str(policy.get("Active", "")).strip().lower() != "yes":
            continue

        effective_date = parse_date(policy.get("Effective From"))

        if not effective_date or effective_date > end_date:
            continue

        from_amount = float(policy.get("From Sales Amount", 0) or 0)
        to_amount = float(policy.get("To Sales Amount", 0) or 0)

        if from_amount <= total_sales <= to_amount:
            if applicable_policy is None or effective_date > applicable_policy[0]:
                applicable_policy = (effective_date, policy)

    if not applicable_policy:
        return 0.0

    policy = applicable_policy[1]
    rate = parse_percent(policy.get("Commission Percent", 0))

    if str(policy.get("Calculation Type", "")).strip() == "Flat":
        return round(total_sales * rate / 100, 2)

    return 0.0


def calculate_attendance(employee_id, start_date, end_date):
    overtime_total = 0
    late_total = 0
    early_total = 0
    violations = []

    for attendance in attendance_data:
        if str(attendance.get("Employee ID", "")).strip() != employee_id:
            continue

        attendance_date = parse_date(attendance.get("Attendance Date"))

        if not attendance_date or not (start_date <= attendance_date <= end_date):
            continue

        day_type = str(attendance.get("Day Type", "")).strip()

        overtime_hours = float(attendance.get("Overtime Hours", 0) or 0)
        late_hours = float(attendance.get("Late Entry Hours", 0) or 0)
        early_hours = float(attendance.get("Early Exit Hours", 0) or 0)

        overtime_policy = get_active_policy(
            attendance_policy_data,
            attendance_date,
            "Overtime",
            day_type
        )

        if overtime_policy and overtime_hours > 0:
            minimum_hours = float(
                overtime_policy.get("Daily Minimum Payable Hours", 0) or 0
            )
            maximum_hours = float(
                overtime_policy.get("Daily Maximum Payable Hours", 0) or 0
            )
            hourly_rate = float(
                overtime_policy.get("Hourly Rate", 0) or 0
            )
            multiplier = float(
                overtime_policy.get("Multiplier", 1) or 1
            )

            payable_hours = overtime_hours

            if overtime_hours < minimum_hours:
                payable_hours = 0

            if maximum_hours and overtime_hours > maximum_hours:
                payable_hours = maximum_hours

                violations.append({
                    "date": str(attendance.get("Attendance Date")),
                    "day_type": day_type,
                    "actual_hours": overtime_hours,
                    "maximum_payable_hours": maximum_hours,
                    "excess_hours": overtime_hours - maximum_hours
                })

            overtime_total += payable_hours * hourly_rate * multiplier

        late_policy = get_active_policy(
            attendance_policy_data,
            attendance_date,
            "Late Entry",
            day_type
        )

        if late_policy and late_hours > 0:
            rate = float(late_policy.get("Hourly Rate", 0) or 0)
            multiplier = float(late_policy.get("Multiplier", 1) or 1)

            late_total += math.ceil(
                late_hours * rate * multiplier
            )

        early_policy = get_active_policy(
            attendance_policy_data,
            attendance_date,
            "Early Exit",
            day_type
        )

        if early_policy and early_hours > 0:
            rate = float(early_policy.get("Hourly Rate", 0) or 0)
            multiplier = float(early_policy.get("Multiplier", 1) or 1)

            early_total += math.ceil(
                early_hours * rate * multiplier
            )

    return {
        "overtime": round(overtime_total, 2),
        "late_deduction": round(late_total, 2),
        "early_deduction": round(early_total, 2),
        "overtime_violations": violations
    }


def verify_payroll(employee_id):
    global payroll_data

    payroll_data = payroll_sheet.get_all_records()

    payroll_row = None

    for row in payroll_data:
        if str(row.get("Employee ID", "")).strip() == employee_id:
            payroll_row = row
            break

    if not payroll_row:
        return {
            "success": False,
            "message": f"No payroll record found for {employee_id}"
        }

    start_date = parse_date(payroll_row.get("Payroll Start Date"))
    end_date = parse_date(payroll_row.get("Payroll End Date"))

    commission = calculate_commission(
        employee_id,
        start_date,
        end_date
    )

    attendance = calculate_attendance(
        employee_id,
        start_date,
        end_date
    )

    expected = {
        "commission": round(commission, 2),
        "overtime": round(attendance["overtime"], 2),
        "total_earnings": round(
            commission + attendance["overtime"], 2
        ),
        "late_deduction": round(
            attendance["late_deduction"], 2
        ),
        "early_deduction": round(
            attendance["early_deduction"], 2
        ),
        "total_deductions": round(
            attendance["late_deduction"] + attendance["early_deduction"], 2
        )
    }

    current = {
        "commission": float(
            payroll_row.get("Commission Earning", 0) or 0
        ),
        "overtime": float(
            payroll_row.get("Overtime Earning", 0) or 0
        ),
        "total_earnings": float(
            payroll_row.get("Total Earnings", 0) or 0
        ),
        "late_deduction": float(
            payroll_row.get("Late Entry Deduction", 0) or 0
        ),
        "early_deduction": float(
            payroll_row.get("Early Exit Deduction", 0) or 0
        ),
        "total_deductions": float(
            payroll_row.get("Total Deductions", 0) or 0
        )
    }

    differences = {}

    for key in expected:
        if round(current[key], 2) != round(expected[key], 2):
            differences[key] = round(
                current[key] - expected[key], 2
            )

    return {
        "success": True,
        "employee_id": employee_id,
        "current": current,
        "expected": expected,
        "differences": differences,
        "discrepancies": bool(differences),
        "overtime_violations": attendance["overtime_violations"]
    }


def update_payroll(employee_id, approved=False):
    result = verify_payroll(employee_id)

    if not result.get("success"):
        return result

    if not result.get("discrepancies"):
        return {
            "success": True,
            "updated": False,
            "message": "No payroll update required."
        }

    if not approved:
        return {
            "success": True,
            "updated": False,
            "message": "Payroll update not approved."
        }

    expected = result["expected"]

    row_number = None
    payroll_row = None

    for index, row in enumerate(payroll_data, start=2):
        if str(row.get("Employee ID", "")).strip() == employee_id:
            row_number = index
            payroll_row = row
            break

    if not row_number:
        return {
            "success": False,
            "message": f"Payroll row not found for {employee_id}"
        }

    payroll_sheet.update(
        f"G{row_number}:L{row_number}",
        [[
            expected["commission"],
            expected["overtime"],
            expected["total_earnings"],
            expected["late_deduction"],
            expected["early_deduction"],
            expected["total_deductions"]
        ]]
    )

    payroll_row["Commission Earning"] = expected["commission"]
    payroll_row["Overtime Earning"] = expected["overtime"]
    payroll_row["Total Earnings"] = expected["total_earnings"]
    payroll_row["Late Entry Deduction"] = expected["late_deduction"]
    payroll_row["Early Exit Deduction"] = expected["early_deduction"]
    payroll_row["Total Deductions"] = expected["total_deductions"]

    return {
        "success": True,
        "updated": True,
        "message": "Payroll updated successfully."
    }


print("Payroll verification and safe update functions are ready.")

Payroll verification and safe update functions are ready.


In [60]:
MODEL = "openrouter/free"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

tools = {
    "verify_payroll": verify_payroll
}

tool_schemas = [
    {
        "type": "function",
        "function": {
            "name": "verify_payroll",
            "description": "Verify an employee payroll record against active commission and attendance policies. Python calculations are authoritative.",
            "parameters": {
                "type": "object",
                "properties": {
                    "employee_id": {
                        "type": "string",
                        "description": "Employee ID to verify"
                    }
                },
                "required": ["employee_id"]
            }
        }
    }
]

system_prompt = """
You are an AI Payroll Verification Agent.

Your role is to coordinate payroll verification and explain the results.

Python calculations and verification results are authoritative.

Do not recalculate, modify, round, or invent payroll numbers.

Use the verify_payroll tool to obtain the payroll verification result.

After receiving the verification result:

1. Explain whether earnings are correct.
2. Explain whether deductions are correct.
3. Identify payroll discrepancies.
4. Identify overtime policy violations.
5. Explain what should be reviewed.

A policy violation does not by itself create a payroll discrepancy.

If payroll values already match the expected values, do not request human approval and do not request a payroll update, even if policy violations are present.

Only payroll discrepancies can trigger human approval.

Do not recommend approval for a policy violation when the payroll values are already correct.

The update_payroll function is controlled by the Python workflow and is not available as an LLM tool.

Keep numerical values exactly as provided by the verification tool.
"""

print("Agent configuration and verification tool registry are ready.")

Agent configuration and verification tool registry are ready.


In [61]:
def verify_and_approve(employee_id):
    result = verify_payroll(employee_id)

    if not result.get("success"):
        print(result.get("message", "Payroll verification failed."))
        return

    prompt = f"""
Verify payroll for employee {employee_id} using the verify_payroll tool.

Verification result:
{result}

Explain the verification result clearly.

If payroll discrepancies are false, state that no approval or payroll update is required, even if policy violations exist.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        tools=tool_schemas
    )

    message = response.choices[0].message

    print("\nAgent Explanation:")
    print(message.content or "Verification completed.")

    print("\nPython Verification Result:")
    print(result)

    if result.get("overtime_violations"):
        print("\nOvertime Policy Violations:")

        for violation in result["overtime_violations"]:
            print(
                f"{violation['date']}: "
                f"{violation['actual_hours']} hours worked, "
                f"maximum payable {violation['maximum_payable_hours']} hours."
            )

    if not result.get("discrepancies"):
        print("\nPayroll Status:")
        print("Payroll is correct. No discrepancies found.")
        return

    print("\nPayroll Status:")
    print("Payroll discrepancies found.")

    approval = input("\nDo you approve updating the payroll? (yes/no): ").strip().lower()

    if approval == "yes":
        update_result = update_payroll(employee_id, approved=True)
        print(update_result["message"])
    else:
        update_result = update_payroll(employee_id, approved=False)
        print(update_result["message"])

In [55]:
verify_and_approve("EMP001")


Agent Explanation:
**Payroll Verification Result for Employee EMP001**

**Earnings:**
- **Commission:** Current $5,000.00 matches expected $5,000.00 → **Correct**
- **Overtime:** Current $8,200.00 matches expected $8,200.00 → **Correct**
- **Total Earnings:** Current $13,200.00 matches expected $13,200.00 → **Correct**

**Deductions:**
- **Late Deduction:** Current $83.00 matches expected $83 → **Correct**
- **Early Deduction:** Current $43.00 matches expected $43 → **Correct**
- **Total Deductions:** Current $126.00 matches expected $126 → **Correct**

**Discrepancies:**
- No differences were found between current and expected values (`differences`: {}).
- The `discrepancies` flag is **false**, meaning the payroll amounts are accurate.

**Policy Violations:**
- An overtime policy violation was identified:
  - **Date:** 6‑Jul‑2026 (Working Day)
  - **Actual hours worked:** 5.0 hours
  - **Maximum payable hours:** 4.0 hours
  - **Excess hours:** 1.0 hour
  - This indicates that on that